# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

### Setup

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

In [ ]:
from langchain_classic import hub
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.tools import Tool
import os
import tiktoken
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

load_dotenv()

# Tavily
tavily_search = TavilySearchResults()

# LLM + Encoding
llm = ChatOpenAI(model="gpt-4", temperature=0)
encoding = tiktoken.encoding_for_model("gpt-4")

# Safe Tavily wrapper
def safe_search(query: str) -> str:
    result = tavily_search.run(query)
    # Ensure result is a string — Tavily returns dict with 'snippets' sometimes
    if isinstance(result, dict):
        result_text = result.get("content", "") or str(result)
    else:
        result_text = str(result)

    tokens = encoding.encode(result_text)
    trimmed = encoding.decode(tokens[:3500])
    return trimmed

# Tool
tools = [
    Tool(
        name="TavilySafeSearch",
        func=safe_search,
        description="Web search tool"
    )
]

# ReAct prompt
prompt = hub.pull("hwchase17/react")

# Create agent
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

# Executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

# Run
agent_executor.invoke({"input": "What is LangChain?"})



> Entering new AgentExecutor chain...
I am not sure what LangChain is. I should use the web search tool to find out.
Action: TavilySafeSearch
Action Input: "What is LangChain?"[{'title': 'What is LangChain?', 'url': 'https://aws.amazon.com/what-is/langchain/', 'content': '## What is LangChain?\n\nLangChain is an open source framework for building applications based on large language models (LLMs). LLMs are large deep-learning models pre-trained on large amounts of data that can generate responses to user queries—for example, answering questions or creating images from text-based prompts. LangChain provides tools and abstractions to improve the customization, accuracy, and relevancy of the information the models generate. For example, developers can use LangChain components to build new prompt chains or customize existing templates. LangChain also includes components that allow LLMs to access new data sets without retraining.\n\nRead about Large Language Models (LLMs)\n\n## Why is Lan

{'input': 'What is LangChain?',
 'output': 'LangChain is an open-source framework for building applications based on large language models. It provides tools and abstractions to improve the customization, accuracy, and relevancy of the information the models generate. It simplifies AI development by abstracting the complexity of data source integrations and prompt refining. It is used to develop LLM-powered applications like intelligent chatbots, sophisticated question-answering systems, and automated data analysis tools.'}

### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [ ]:
# Your prompt here
prompt_1 = """
Summarize the latest dadvancements in generative AI applied to healthcare in 2025 in Markdown format with:
- Main heading: applied to healthcare in 2025
- Subheadings per key model or development
- Bullet points for features
- Source links if any
Keep it concise.

"""
# Run it
response_1 = agent_executor.invoke({"input": prompt_1})
print(response_1["output"])




> Entering new AgentExecutor chain...
I need to search for the latest advancements in generative AI applied to healthcare in 2025. I will use the TavilySafeSearch tool to find this information.
Action: TavilySafeSearch
Action Input: "latest advancements in generative AI applied to healthcare in 2025"[{'title': 'Generative AI in Healthcare: Use Cases, Benefits, Challenges of GenAI and Trends 2025', 'url': 'https://www.johnsnowlabs.com/generative-ai-healthcare/', 'content': 'Generative AI in healthcare is advancing various practical applications. One prominent use case is the implementation of Retrieval-Augmented Generation (RAG) on FHIR (Fast Healthcare Interoperability Resources), which enables healthcare systems to make data more accessible by pairing RAG with FHIR. This approach, developed using John Snow Labs’ language models, allows real-time retrieval of data based on specific questions, rather than relying on pre-trained models, making healthcare data more transparent and easil

In [9]:
display(Markdown(response_1["output"]))


# Applied to Healthcare in 2025

## Retrieval-Augmented Generation (RAG) on FHIR
- Enables healthcare systems to make data more accessible
- Allows real-time retrieval of data based on specific questions
- Makes healthcare data more transparent and easily interpretable
- [Source](https://www.johnsnowlabs.com/generative-ai-healthcare/)

## FunctionalMind by John Snow Labs
- Assists clinicians in applying the latest evidence-based medicine
- Analyzes vast medical knowledge bases
- Summarizes the latest clinical research
- Provides actionable insights for patient care
- [Source](https://www.johnsnowlabs.com/generative-ai-healthcare/)

## GenAI Models by Google
- Used for imaging interpretation, clinical documentation, and medical copilots for physicians
- [Source](https://www.crescendo.ai/news/ai-in-healthcare-news)

## Abridge’s Ambient Documentation Solution by Kaiser Permanente
- Deployed across 40 hospitals and 600+ medical offices
- Marked the largest generative AI rollout in healthcare history
- Reduced documentation time by more than 50%
- Automated prior authorizations, referrals, and coding workflows
- [Source](https://menlovc.com/perspective/2025-the-state-of-ai-in-healthcare/)

### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [12]:
# Your prompt here
prompt_2 = """
You are a research assistant. Use web search to find the most notable **2025 AI startups** and their key innovations.

Return a clean, structured Markdown report with:

- Top Emerging AI Startups in 2025

For each startup include:

- Company Name
- Product / Innovation (2025)with 1 or 2 bullets
- Why its notable in 2025
- Source link

Rules:
- Keep it concise and well-structured in Markdown.
"""

# Run it
response_2 = agent_executor.invoke({"input": prompt_2})
print(response_2["output"])




> Entering new AgentExecutor chain...
I need to use the web search tool to find information about the most notable AI startups in 2025 and their key innovations. I will then structure this information into a Markdown report.

Action: TavilySafeSearch
Action Input: "Top AI startups in 2025 and their key innovations"
[{'title': 'Top 25 AI startups in 2025: OpenAI, xAI, Perplexity, Cohere ...', 'url': 'https://www.linkedin.com/posts/yrebryk_these-are-the-top-25-most-valuable-ai-startups-activity-7387111846970908672-kPS_', 'content': "as a true Google challenger. 4️⃣ Cohere extended its round with $100M, now valued at $7B, doubling down on secure enterprise AI. 5️⃣ ElevenLabs received a strategic Nvidia investment, scaling its AI voice synthesis tech after a $3.3B valuation earlier this year. 📊 Key insights from the list: • 🇺🇸 17 of 26 startups (65%) are US-based • 🧠 10 (38%) focus on LLMs or foundational models • 🦄 Most reached unicorn status in under 2 years • ⏱️ Average company age: ~

In [13]:
display(Markdown(response_2["output"]))


# Top Emerging AI Startups in 2025

## OpenAI
- Product / Innovation (2025): Shipping more accessible toolchains that simplify fine-tuning for industry apps.
- Why it's notable in 2025: OpenAI has a $500B valuation, making it one of the most valuable AI startups.
- [Source link](https://www.linkedin.com/posts/yrebryk_these-are-the-top-25-most-valuable-ai-startups-activity-7387111846970908672-kPS_)

## Anthropic
- Product / Innovation (2025): Safety-first approaches are maturing in practical deployments.
- Why it's notable in 2025: Anthropic has raised $13B, with a valuation of approximately $183B.
- [Source link](https://www.linkedin.com/posts/yrebryk_these-are-the-top-25-most-valuable-ai-startups-activity-7387111846970908672-kPS_)

## xAI (Elon Musk)
- Product / Innovation (2025): Not specified.
- Why it's notable in 2025: xAI has raised $10B.
- [Source link](https://www.linkedin.com/posts/yrebryk_these-are-the-top-25-most-valuable-ai-startups-activity-7387111846970908672-kPS_)

## Cohere
- Product / Innovation (2025): Doubling down on secure enterprise AI and global rollout.
- Why it's notable in 2025: Cohere extended its funding with $100M, reaching a $7B valuation.
- [Source link](https://www.linkedin.com/posts/yrebryk_these-are-the-top-25-most-valuable-ai-startups-activity-7387111846970908672-kPS_)

## ElevenLabs
- Product / Innovation (2025): Scaling up their AI voice technology.
- Why it's notable in 2025: ElevenLabs received a strategic investment from Nvidia, following a $3.3B valuation earlier this year.
- [Source link](https://www.linkedin.com/posts/yrebryk_these-are-the-top-25-most-valuable-ai-startups-activity-7387111846970908672-kPS_)

## Hugging Face
- Product / Innovation (2025): Expands open models and pipelines that democratize deployment.
- Why it's notable in 2025: Hugging Face has a $4.5B valuation.
- [Source link](https://www.linkedin.com/posts/theravitshow_top-30-ai-startups-shaping-the-future-of-activity-7394864149270478848-aOoj)

## Mistral AI
- Product / Innovation (2025): Not specified.
- Why it's notable in 2025: Mistral AI has a $6B valuation.
- [Source link](https://www.linkedin.com/posts/yrebryk_these-are-the-top-25-most-valuable-ai-startups-activity-7387111846970908672-kPS_)

## Thinking Machines Lab
- Product / Innovation (2025): Developing broadly capable AI systems.
- Why it's notable in 2025: Thinking Machines Lab is reportedly raising $1 billion at a valuation of about $9 billion.
- [Source link](https://www.forbes.com/lists/ai50/)

## World Labs
- Product / Innovation (2025): Developing models that can understand physical spaces.
- Why it's notable in 2025: World Labs has raised $291.5 million.
- [Source link](https://www.forbes.com/lists/ai50/)

## Writer
- Product / Innovation (2025): Training its own models to handle mundane business tasks like writing marketing blogs and searching through troves of documents.
- Why it's notable in 2025: Writer has raised $326 million.
- [Source link](https://www.forbes.com/lists/ai50/)

### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [17]:
# Your prompt here
prompt_3 = """
Compare ChatGPT Team and Claude Pro.

Return a Markdown comparison with:
ChatGPT Team vs Claude Pro

Features:
- Bullet points per product

Pricing:
- Monthly price and plan notes

Reviews:
- 1 or 2 summarized user or expert opinions per product
- Include source links

Keep it concise and factual.

"""

# Run it
response_3 = agent_executor.invoke({"input": prompt_3})
print(response_3["output"])




> Entering new AgentExecutor chain...
I need to find information about the features, pricing, and reviews of both ChatGPT Team and Claude Pro. I will start by searching for information about ChatGPT Team.
Action: TavilySafeSearch
Action Input: ChatGPT Team features pricing reviews[{'title': 'Is ChatGPT Teams right for your business? An honest 2025 review', 'url': 'https://www.eesel.ai/blog/chatgpt-teams', 'content': "| Feature | ChatGPT Plus | ChatGPT Teams |\n --- \n| Price | $20/user/month | $30/user/month (monthly) or $25/user/month (annually) |\n| Users | 1 | 2+ |\n| Workspace | Personal | Shared Team Workspace |\n| Admin Controls | No | Yes |\n| Data Training | Opt-out available | Never trained on your data by default |\n| Custom GPTs | Create for personal use | Create and share within workspace |\n| Early Access | Often receives new features first | Can lag behind the Plus plan |\n\n## So, Is ChatGPT Teams the right choice? [...] $30 per user, per month if you pay monthly.\n $2

In [18]:
display(Markdown(response_3["output"]))

# ChatGPT Team vs Claude Pro

## Features:

**ChatGPT Team:**
- Shared team workspace
- Admin controls
- Data privacy guarantees
- Custom GPTs creation and sharing within workspace
- Access to image generation capabilities using the latest DALL-E 3 model
- Access to advanced data analysis capabilities
- Access to web browsing
- Image and voice input and output
- [Source](https://www.eesel.ai/blog/chatgpt-teams)

**Claude Pro:**
- Access to the advanced Opus model
- Higher usage limits
- Models beyond Sonnet 4
- Unlimited Projects for organizing chats and documents
- Access to Claude in your terminal, deep research, extended thinking for complicated tasks, full integration support
- [Source](https://claude.com/pricing)

## Pricing:

**ChatGPT Team:**
- $30/user/month (monthly)
- $25/user/month (annually)
- [Source](https://www.eesel.ai/blog/chatgpt-teams)

**Claude Pro:**
- $20/user/month (monthly)
- $17/user/month (annually)
- [Source](https://claude.com/pricing)

## Reviews:

**ChatGPT Team:**
- Users have noted that any team member, not just an admin, can invite new people to the workspace which can lead to a surprisingly high bill at the end of the month. [Source](https://www.eesel.ai/blog/chatgpt-teams)
- The price covers access to the shared workspace, admin controls, higher usage caps on the best models, custom GPTs, and the data privacy guarantees that businesses need. [Source](https://www.eesel.ai/blog/chatgpt-teams)

**Claude Pro:**
- If you’re a heavy user (more than 20 interactions per day), the Pro plan offers a very high return on investment. [Source](https://www.gamsgo.com/blog/is-claude-pro-worth-it)
- Claude Pro gives access to the advanced Opus model, offering stronger coding, reasoning, and long-context performance. [Source](https://www.gamsgo.com/blog/is-claude-pro-worth-it)

## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.
